In [0]:
print("=" * 80)
print("GOLD LAYER - BUSINESS METRICS & AGGREGATIONS")
print("=" * 80)

# ============ READ SILVER TABLES ============
print("\nReading Silver tables")
df_silver_csv = spark.table("dm2_credit_risk.silver_loan_apps")
df_silver_json = spark.table("dm2_credit_risk.silver_world_bank")

print(f"silver_loan_apps: {df_silver_csv.count():,} rows")
print(f"silver_world_bank: {df_silver_json.count():,} rows")

# ============ CREATE GOLD LAYER WITH SQL ============
print("\n" + "=" * 80)
print("CREATING GOLD LAYER TABLES")
print("=" * 80)



In [0]:
%sql

CREATE OR REPLACE TABLE dm2_credit_risk.gold_credit_risk_metrics AS
WITH loan_with_country AS (
    SELECT *,
        CASE 
            WHEN AMT_INCOME_TOTAL > 200000 THEN 'USA'
            WHEN AMT_INCOME_TOTAL > 150000 THEN 'DEU'
            WHEN AMT_INCOME_TOTAL > 100000 THEN 'GBR'
            WHEN AMT_INCOME_TOTAL > 70000 THEN 'FRA'
            WHEN AMT_INCOME_TOTAL > 50000 THEN 'BRA'
            WHEN AMT_INCOME_TOTAL > 30000 THEN 'CHN'
            ELSE 'IND'
        END as assigned_country
    FROM dm2_credit_risk.silver_loan_apps
),
latest_gdp AS (
    SELECT country_code, gdp_usd
    FROM dm2_credit_risk.silver_world_bank
    WHERE year = (SELECT MAX(year) FROM dm2_credit_risk.silver_world_bank)
)
SELECT 
    CASE 
        WHEN l.credit_to_income_ratio > 3 THEN 'High Risk'
        WHEN l.credit_to_income_ratio > 1.5 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END as risk_segment,
    l.family_status,
    l.education_type,
    l.assigned_country,
    g.gdp_usd as country_gdp_usd,
    COUNT(l.SK_ID_CURR) as total_customers,
    SUM(CASE WHEN l.TARGET = 1 THEN 1 ELSE 0 END) as defaulted_customers,
    ROUND(SUM(CASE WHEN l.TARGET = 1 THEN 1 ELSE 0 END) / COUNT(l.SK_ID_CURR) * 100, 2) as default_rate_percent,
    ROUND(AVG(l.AMT_CREDIT), 2) as avg_credit_amount,
    ROUND(AVG(l.AMT_INCOME_TOTAL), 2) as avg_income,
    ROUND(AVG(l.credit_to_income_ratio), 2) as avg_credit_to_income_ratio,
    COUNT(DISTINCT l.SK_ID_CURR) as unique_customers
FROM loan_with_country l
LEFT JOIN latest_gdp g ON l.assigned_country = g.country_code
GROUP BY risk_segment, l.family_status, l.education_type, l.assigned_country, g.gdp_usd
ORDER BY default_rate_percent DESC;

SELECT * FROM dm2_credit_risk.gold_credit_risk_metrics LIMIT 10;